# Heterogeneity & Forest Plots — Runnable Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anacodicai/learn-notebooks/blob/main/notebooks/evidence-synthesis/15-1/heterogeneity-and-forest-plots.ipynb) &nbsp; [View on GitHub](https://github.com/anacodicai/learn-notebooks/blob/main/notebooks/evidence-synthesis/15-1/heterogeneity-and-forest-plots.ipynb)

> Companion to the **Heterogeneity & Forest Plots** article. Quantify disagreement with **Cochran's Q**, **I²**, and **τ²**, then render a **text forest plot** (boxes sized by weight, a no-effect line, a pooled diamond) — no matplotlib, pure ASCII.

## What you'll build

1. `heterogeneity` — **Q**, **df**, **I²**, and **τ²** from the same four studies.
2. `forest_plot` — a text forest plot on a log axis, exponentiating ratios for display.
3. A run that reproduces the article's Q = 6.81, I² = 56%, τ² = 0.058 and the plotted per-study CIs — pure `numpy`, **no API key**.

### Setup (numpy only)

In [ ]:
import numpy as np

Z = 1.96   # 95% two-sided normal multiplier
print("ready — Z =", Z)

## The same four studies

We reuse the meta-analysis dataset (odds ratios on the log scale) so the heterogeneity numbers line up with the previous notebook.

**Cochran's Q** is a weighted sum of squared deviations from the pooled effect; compared to `df = k − 1`, an excess signals heterogeneity. **I²** rescales that into a percentage; **τ²** is the between-study variance in the effect's own units.

In [ ]:
labels = ['Study 1', 'Study 2', 'Study 3', 'Study 4']
ORs    = np.array([0.60, 0.70, 0.40, 0.90])
theta  = np.array([-0.51, -0.36, -0.92, -0.11])   # ln(OR), 2-dp (matches the article table)
se     = np.array([0.20, 0.25, 0.30, 0.15])
print("ORs :", ORs)
print("theta = ln(OR):", theta)

## The heterogeneity function

```
Q   = Σ w_i (θ_i − θ_pooled)²          w_i = 1/SE_i²,  df = k − 1
I²  = max(0, (Q − df)/Q) · 100%
τ²  = max(0, (Q − df)/(Σw − Σw²/Σw))
```

The `max(0, …)` floor means τ² is never negative: if `Q < df`, τ² = 0 and random-effects collapses back to fixed-effect.

In [ ]:
def heterogeneity(theta, se):
    """Return Q, df, I2 (%), tau2 and the fixed pooled effect (analysis scale)."""
    theta, se = np.asarray(theta, float), np.asarray(se, float)
    w = 1.0 / se**2
    theta_pooled = np.sum(w * theta) / np.sum(w)
    Q = np.sum(w * (theta - theta_pooled)**2)
    df = len(theta) - 1
    I2 = max(0.0, (Q - df) / Q) * 100 if Q > 0 else 0.0
    denom = np.sum(w) - np.sum(w**2) / np.sum(w)
    tau2 = max(0.0, (Q - df) / denom) if denom > 0 else 0.0
    return Q, df, I2, tau2, theta_pooled

Q, df, I2, tau2, tp = heterogeneity(theta, se)
print(f"Q   = {Q:.2f}  (df = {df})")
print(f"I2  = {I2:.1f}%   -> {'high' if I2>=75 else 'moderate' if I2>=50 else 'low'} heterogeneity")
print(f"tau2= {tau2:.4f}   (between-study variance on the log-OR scale)")
print(f"tau = {np.sqrt(tau2):.3f}   (between-study SD)")

## The forest plot — element by element

One row per study: a **box** at the estimate (marked `#`), a **line** for the 95% CI, and a vertical **no-effect line** at OR = 1. The bottom **diamond** (`<>`) is the pooled random-effects estimate. Per-study rows show **fixed-effect** weights while the diamond uses **random-effects** — so you *see* how τ² redistributes influence.

In [ ]:
def forest_plot(labels, theta, se, ratio=True, lo_ax=0.2, hi_ax=5.0, width=40):
    """Render a text forest plot. Ratios are exponentiated and drawn on a log axis."""
    theta, se = np.asarray(theta, float), np.asarray(se, float)
    w = 1.0 / se**2
    weights = w / w.sum() * 100

    # random-effects pooled estimate (so the diamond reflects heterogeneity)
    Q, df, I2, tau2, _ = heterogeneity(theta, se)
    w_star = 1.0 / (se**2 + tau2)
    tp = np.sum(w_star * theta) / np.sum(w_star)
    se_p = np.sqrt(1.0 / np.sum(w_star))

    loglo, loghi = np.log(lo_ax), np.log(hi_ax)
    def col(x):                                    # map a log-scale value to a column
        return int(round((np.log(x) - loglo) / (loghi - loglo) * (width - 1)))

    def row(est, lo, hi, mark="#"):
        line = [" "] * width
        for cc in range(col(lo), col(hi) + 1):     # CI line
            if 0 <= cc < width: line[cc] = "-"
        line[max(0, min(width-1, col(est)))] = mark
        return "".join(line)

    noeff = col(1.0) if ratio else col(0.0)
    print(f"{'study':<10} {'':<{width}}  effect (95% CI)      weight")
    for lab, t, s, wt in zip(labels, theta, se, weights):
        est = np.exp(t) if ratio else t
        lo, hi = np.exp(t - Z*s), np.exp(t + Z*s)
        print(f"{lab:<10} {row(est, lo, hi):<{width}}  "
              f"{est:.2f} ({lo:.2f}-{hi:.2f})   {wt:4.1f}%")

    noeff_line = "".join("|" if cc == noeff else "." for cc in range(width))
    print(f"{'':<10} {noeff_line}  <- no-effect line")
    est_p = np.exp(tp) if ratio else tp
    lo_p, hi_p = np.exp(tp - Z*se_p), np.exp(tp + Z*se_p)
    print(f"{'POOLED':<10} {row(est_p, lo_p, hi_p, mark='<>'):<{width}}  "
          f"{est_p:.2f} ({lo_p:.2f}-{hi_p:.2f})  100.0%")
    print(f"\nQ={Q:.2f} (df={df})  I2={I2:.0f}%  tau2={tau2:.4f}  -> "
          f"{'high' if I2>=75 else 'moderate' if I2>=50 else 'low'} heterogeneity")

print("forest_plot defined")

### Draw it

In [ ]:
print(f"Q={Q:.2f}  df={df}  I2={I2:.1f}%  tau2={tau2:.4f}\n")
forest_plot(labels, theta, se, ratio=True)

## How to read what just printed

- **Study 4** (`0.90`, CI 0.67–1.20) sits right of the others and its CI **crosses 1** — non-significant on its own and the visible source of the heterogeneity; its box carries **46%** of the fixed weight, warning it pulls the pool toward 'no effect'.
- **Studies 1 and 3** sit left (clear benefit); **study 2** straddles the line.
- The **diamond** (pooled RE OR ≈ 0.66) is left of 1 and does **not** cross it → the pooled effect is significant.
- Rows that don't line up = disagreement, which the **I² = 56%** confirms.

## Recap

- **Heterogeneity** is real variation in true effects — disagreement beyond sampling noise.
- **Q = 6.81** vs **df = 3** (more than double) → studies disagree; **I² = 56%** → moderate; **τ² = 0.058** is the spread random-effects folds into the CI.
- The **forest plot** shows it: boxes sized by weight, a no-effect line, a pooled diamond.
- **High I²** → use random-effects, investigate with subgroups / meta-regression, run leave-one-out sensitivity, consider a prediction interval.
- Report Q, I², **and** τ² together, and always **show the plot**.

← Back to the **Heterogeneity & Forest Plots** article. See also **2×2 Tables & Effect Measures** and **Meta-Analysis: Pooling Studies**.